In [2]:
import os
import glob
import shutil

import numpy as np
import cv2
import matplotlib.pyplot as plt
import yaml
from tqdm.contrib.concurrent import thread_map# or thread_map

import alif_awb
import utils_cam4_revb

%load_ext autoreload
%autoreload 2

In [5]:
BG_FOLDER = "/home/klymy/Pictures/background1/4.2_background"
YOLO_DATASET_PATH = "/home/klymy/data/cam4_revB/yolo_dataset"
DATASET_NAME_0 = "batch8_10_margin_15_wb_alif"
DATASET_NAME_1 = "batch8_10_margin_15_wb_alif_clahe"

img_paths = glob.glob(os.path.join(BG_FOLDER, "*.tif"))
len(img_paths)

25

In [6]:
def process_file(pth):
    img = cv2.imread(pth)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = alif_awb.white_balance_vectorized(img_rgb)
    res = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)
    cv2.imwrite(pth, res)

    return 1

def process_file_clahe(pth):
    img = cv2.imread(pth)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    res = alif_awb.white_balance_vectorized(img_rgb)
    res = utils_cam4_revb.clahe(res)
    res = cv2.cvtColor(res, cv2.COLOR_RGB2BGR)
    cv2.imwrite(pth, res)

    return 1

In [11]:
img_names = []
for img_pth in img_paths:
    img_name = os.path.basename(img_pth)
    img_name = img_name.replace(".tif", ".jpg")
    img_names.append(img_name)
    
    yolopath_0 = os.path.join(YOLO_DATASET_PATH, DATASET_NAME_0, "images", "train", img_name)
    yolopath_1 = os.path.join(YOLO_DATASET_PATH, DATASET_NAME_1, "images", "train", img_name)
    
    shutil.copy(img_pth, yolopath_0)
    shutil.copy(img_pth, yolopath_1)

    process_file(yolopath_0)
    process_file_clahe(yolopath_1)



In [14]:
def append_file_names(pth, img_names):
    for img_name in img_names:
        # print(os.path.join(pth, img_name.replace(".jpg", ".txt")))
        with open(os.path.join(pth, img_name.replace(".jpg", ".txt")), "a") as f:
            pass

file_list_path_0 = os.path.join(YOLO_DATASET_PATH, DATASET_NAME_0, "labels", "train")
file_list_path_1 = os.path.join(YOLO_DATASET_PATH, DATASET_NAME_1, "labels", "train")

append_file_names(file_list_path_0, img_names)
append_file_names(file_list_path_1, img_names)